In [ ]:
# 修改预训练模型的缓存目录
import os

os.environ["MODELSCOPE_CACHE"] = r"G:\code\pretrain_model_dir\_modelscope"

In [ ]:
!modelscope download --dataset InfiniAI/LVEval --local_dir G:\code\pretrain_model_dir\_modelscope\LVEval

In [ ]:
!modelscope download --dataset swift/TextCaps --local_dir G:\code\pretrain_model_dir\_modelscope\TextCaps

In [ ]:
!modelscope download --model iic/gme-Qwen2-VL-2B-Instruct --local_dir G:\code\pretrain_model_dir\_modelscope\gme-Qwen2-VL-2B-Instruct

In [ ]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download("Qwen/Qwen3-0.6B")
model_dir

In [ ]:
model_dir

In [ ]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download("Qwen/Qwen3-8B")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = r"G:\code\pretrain_model_dir\_modelscope\qwen\Qwen3-8B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
model.device, model.dtype, model.num_parameters()

In [ ]:
# prepare the model input
prompt = "我是一个服装设计师, 想为鞠婧祎设计衣服, 需要了解她的三围"
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

In [ ]:
print(text)
text

In [ ]:
len(output_ids)